<a href="https://colab.research.google.com/github/dohee-jin/data-mining-final-project/blob/main/notebooks/kbo_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# KBO 골든글러브 후보 예측 분석

## 1. 골든글러브 후보 예측

- 선수 개인 기록을 활용해 2026년 골든글러브 후보 가능성이 높은 선수를 예측한다.
- 타자와 투수는 기록 컬럼이 다르므로 분리해서 분석한다.

---

## 2. 골든글로브 후보 예측 데이터 사용 전략

데이터는 크게 세 종류로 나눈다.

| 구분 | 출처 | 직접 정리 여부 | 사용 목적 |
|---|---|---:|---|
| 과거 선수 기록 | Kaggle KBO Player Dataset 1982-2025 | X | 골든글러브 예측 학습 데이터 |
| 2026 현재 선수 기록 | KBO 공식 홈페이지 기록실 | O | 2026 골든글러브 후보 예측 대상 |
| 골든글러브 수상자 목록 | KBO 공식 골든글러브 수상 현황 | O | `is_goldenglove` 라벨 생성 |

---

## 3. 골든글러브 후보 예측용 데이터

### 3.1 Kaggle에서 가져올 데이터

Kaggle의 `KBO Player Dataset (1982-2025)`를 과거 학습 데이터로 사용한다.

사용할 데이터는 다음과 같다:

```
kbo_batting_stats_by_season_1982-2025.csv
kbo_pitching_stats_by_season_1982-2025.csv
```

Kaggle 데이터는 1982년부터 2025년까지의 KBO 정규시즌 선수 타격 및 투수 기록을 포함한다.

**추천 사용 기간:** 2015~2025

### 3.2 직접 정리해야 하는 2026 선수 기록

2026년은 아직 시즌이 진행 중이므로 Kaggle 과거 데이터에 포함되어 있지 않다. 따라서 KBO 공식 홈페이지에서 현재 기록을 복사해 CSV로 직접 정리해야 한다.

```
data/raw/kbo_hitter_data_2026.csv
data/raw/kbo_pitcher_data_2026.csv
```

2026 선수 기록 파일에는 반드시 `year` 컬럼을 추가한다. 타자 기록에는 골든글러브 포지션별 예측을 위해 `position` 칼럼을 추가한다.

**예시:**

| year | player | team | avg | hr | rbi | ops |
|---:|---|---|---:|---:|---:|---:|
| 2026 | 선수명 | LG | 0.315 | 12 | 45 | 0.890 |

### 3.3 kaggle 데이터셋 접근 불가

kaggle 데이터셋이 내려가 접근하지 못해 해당 데이터셋으로의 모델 생성, 예측은 불가능하게 됨. 

Hugging Face의 `juhonov/KBOresearch` 데이터셋을 통해 2000-2025년도 선수 기록을 바탕으로 데이터를 수집, 학습한다.

---

## 데이터 로드

In [ ]:
!pip install -q datasets

from huggingface_hub import hf_hub_download
import pandas as pd
import json

print('필요한 라이브러리 임포트 완료')

In [ ]:
# Hugging Face에서 JSON 파일 직접 다운로드
print('Hugging Face에서 데이터 다운로드 중...')

hitter_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_hitter_stats_2000_2025.json",
    repo_type="dataset"
)

pitcher_file = hf_hub_download(
    repo_id="juhonov/KBOresearch",
    filename="kbo_pitcher_stats_2000_2025.json",
    repo_type="dataset"
)

print(f"타자 데이터 경로: {hitter_file}")
print(f"투수 데이터 경로: {pitcher_file}")

In [ ]:
# JSON 파일을 pandas DataFrame으로 변환
print('JSON 파일 파싱 중...')

with open(hitter_file, 'r', encoding='utf-8') as f:
    hitter_data = json.load(f)

with open(pitcher_file, 'r', encoding='utf-8') as f:
    pitcher_data = json.load(f)

df_hitter = pd.DataFrame(hitter_data)
df_pitcher = pd.DataFrame(pitcher_data)

print(f"타자 데이터 크기: {df_hitter.shape}")
print(f"투수 데이터 크기: {df_pitcher.shape}")

In [ ]:
print("\n=== 타자 데이터 컬럼 ===")
print(df_hitter.columns.tolist())

print("\n=== 투수 데이터 컬럼 ===")
print(df_pitcher.columns.tolist())

In [ ]:
print("\n=== 타자 데이터 샘플 ===")
print(df_hitter.head())

print("\n=== 투수 데이터 샘플 ===")
print(df_pitcher.head())

In [ ]:
print("\n=== 타자 데이터 정보 ===")
print(df_hitter.info())

print("\n=== 투수 데이터 정보 ===")
print(df_pitcher.info())